# Tool calling mode
Turn the Pydantic schema into the parameters of a "fake" function and force the model to call it.

In [1]:
import json
import time

from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import JSON, Markdown, display

load_dotenv()  # loads GEMINI_API_KEY from .env
client = genai.Client()

from demo_helpers import VALID_APPLICATIONS

In [2]:
transcript = "Hey Google, navigate to Ara."
TOOL_NAME = "submit_parsed_command"

In [9]:
from pydantic import BaseModel, Field


class ParsedCommand(BaseModel):
    target_application: str = Field(description="The application the user wants to use")
    target_action: str = Field(
        description="The action to perform within that application"
    )
    additional_parameters: dict[str, str] = Field(
        default_factory=dict,
        description=(
            "Parameters for the action mentioned in the transcript, as a mapping "
            "of parameter name (as listed for the chosen action) to value"
        ),
    )


print(json.dumps(ParsedCommand.model_json_schema(), indent=2))

{
  "properties": {
    "target_application": {
      "description": "The application the user wants to use",
      "title": "Target Application",
      "type": "string"
    },
    "target_action": {
      "description": "The action to perform within that application",
      "title": "Target Action",
      "type": "string"
    },
    "additional_parameters": {
      "additionalProperties": {
        "type": "string"
      },
      "description": "Parameters for the action mentioned in the transcript, as a mapping of parameter name (as listed for the chosen action) to value",
      "title": "Additional Parameters",
      "type": "object"
    }
  },
  "required": [
    "target_application",
    "target_action"
  ],
  "title": "ParsedCommand",
  "type": "object"
}


## Input

In [4]:
message = f"""For the following transcript, determine the target action the user wants to take,
the target application, and any additional parameters.\n
The target application, action and parameters must be chosen from the following
valid applications (each lists its actions and the parameters each action accepts):\n
{json.dumps(VALID_APPLICATIONS, indent=2)}\n
Submit your answer by calling the {TOOL_NAME} tool.

transcript: {transcript}"""
print(message)

For the following transcript, determine the target action the user wants to take,
the target application, and any additional parameters.

The target application, action and parameters must be chosen from the following
valid applications (each lists its actions and the parameters each action accepts):

{
  "maps": {
    "description": "Navigation and location search",
    "actions": {
      "navigate": {
        "destination": "str",
        "mode": "driving | walking | cycling | transit"
      },
      "search_nearby": {
        "query": "str",
        "radius_km": "float"
      },
      "show_traffic": {}
    }
  },
  "music": {
    "description": "Music streaming and playback",
    "actions": {
      "play": {
        "song": "str",
        "artist": "str",
        "playlist": "str"
      },
      "pause": {},
      "skip": {},
      "set_volume": {
        "level": "int (0-100)"
      }
    }
  },
  "messages": {
    "description": "Send and read text messages",
    "actions": {
   

In [10]:
submit_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name=TOOL_NAME,
            description="Submit the parsed command extracted from the transcript.",
            parameters_json_schema=ParsedCommand.model_json_schema(),
        )
    ]
)

config = types.GenerateContentConfig(
    tools=[submit_tool],
    # Force the model to call our tool (no free-text answer allowed)
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(
            mode="ANY",
            allowed_function_names=[TOOL_NAME],
        )
    ),
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)
config

GenerateContentConfig(
  automatic_function_calling=AutomaticFunctionCallingConfig(
    disable=True,
    maximum_remote_calls=10
  ),
  tool_config=ToolConfig(
    function_calling_config=FunctionCallingConfig(
      allowed_function_names=[
        'submit_parsed_command',
      ],
      mode=<FunctionCallingConfigMode.ANY: 'ANY'>
    )
  ),
  tools=[
    Tool(
      function_declarations=[
        FunctionDeclaration(
          description='Submit the parsed command extracted from the transcript.',
          name='submit_parsed_command',
          parameters_json_schema={
            'properties': {<... 3 items at Max depth ...>},
            'required': [<... 2 items at Max depth ...>],
            'title': 'ParsedCommand',
            'type': 'object'
          }
        ),
      ]
    ),
  ]
)

## Model call

In [19]:
start = time.perf_counter()
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=message,
    config=config,
)
print(f"Client call took {time.perf_counter() - start:.2f}s")

Client call took 1.77s


## Output

In [20]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'additional_parameters': {<... 1 item at Max depth ...>},
            'target_action': 'navigate',
            'target_application': 'maps'
          },
          id='call_7664628859057267009',
          name='submit_parsed_command'
        ),
        thought_signature=b"\x12`\n^\x01i\x14}\x13V\xf6\x15F\xbb\x976\xd9\x1f,)X\xc9\xd1W\xd2\x88%[\tg\xce\xd7\xe7\x97\xff\x92\x1e9\xe4\xca\xb1\x8fdE\n\xfbKx\x1dEi<\xf2\x84V\xdc8\x03\xe2'<\xfe\xf8\xd9\x1c \x9e\xe4$\xa6A\xe4\x11d\xf3\xa6{'\x9a\xa9\xf4\xd2\xefN[\xc9\x02\xfa\xd5\xac\x9b\xcd\x82\xc8\xf6X\x19\xfe"
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='tsn3ap6iF5z_z7sPy8eb0Qk' usage_metadata=GenerateContentResponseUsageMet

In [22]:
function_call = response.function_calls[0]
function_call

FunctionCall(
  args={
    'additional_parameters': {
      'destination': 'Ara'
    },
    'target_action': 'navigate',
    'target_application': 'maps'
  },
  id='call_7664628859057267009',
  name='submit_parsed_command'
)

In [23]:
# Validate the tool-call arguments against the Pydantic model
result = ParsedCommand.model_validate(function_call.args)
result

ParsedCommand(target_application='maps', target_action='navigate', additional_parameters={'destination': 'Ara'})